In [14]:
# ===== Task 2: Apply dimension reduction techniques (PCA + KNN) =====
# This is the code for task 2. As per requirements, we are using sklearn packages here.
# Also forgot to mention that if you are ever running this locally, ensure that you have installed numpy, pandas, matplotlib as well as sklearn in your venv
# One thing that caught me out, the pip package is called scikit-learn and not sklearn. sklearn on its own is a dead stub package that just errors out.
# Aim is to use PCA to reduce the dimension of the features.

# What the task is asking for:
# 2a, code implementation of PCA on the train and test sets
# 2b, report the macro f1 for 2000, 1000, 500 and 100 components on the test set
# we cannot work the macro f1 out ourselves because we do not have the test
# labels, so it has to come from submitting to kaggle and reading the score off
# the leaderboard. the model is fixed by the brief, KNN with n_neighbors=2, so
# there is nothing for us to tune. the only thing that changes between the four
# runs is the number of components, which is the whole point of the experiment.

# From deliverables:
# Need a trained KNN model for its predictions on test after being trained
# Need a test data in PCA space
# Need PCA-reduced train features
# Need train features, labels, test id's as well as X_test
# Need loading of the x, y, X_test, test_ids
# Need shape and id checks
# Need to fit the PCA on train only


In [15]:
# First starting off with a bit of data analysis to get to know what we are working with here
# Need to check for attributes that would be specific for PCA and therefore would influence the later process
# There are no features that dominate through unit choice, so we dont standardise
# Basic concept here is that PCA is going to try and retain as much variance in the data while it simplifies the dataset as muchas possible

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

DATA = "50-007-machine-learning-may-2026/"

# This has the same layout as task 1 where train_features.csv has id, label and test_features has id
# Load dataset
train_df = pd.read_csv(DATA + "train_features.csv")
test_df = pd.read_csv(DATA + "test_features.csv")

# .values pulls the raw numpy array out of the df since sklearn needs numbers and not columns. This is being used by the PCA as well as KNN as they are linear algrbra
X = train_df.drop(columns=["id","label"]).values     # These are the train features
y = train_df["label"].values                         # These are the labels
X_test = test_df.drop(columns=["id"]).values         # This is the X_test
test_ids = test_df["id"].values                      # These are the test_ids           

# Check the dataframe
print("X       ", X.shape)
print("X_test  ", X_test.shape)
print("dtype   ", X.dtype)



X        (20000, 5000)
X_test   (6999, 5000)
dtype    float64


In [16]:
# Fit the PCA once at the biggest k we need, instead of fitting it four times.
# the components come out already sorted best first, and that ordering does not
# depend on how many we asked for. so the first 100 columns of a 2000 component
# fit are the same thing as fitting with 100 components from scratch. I check
# that in the next cell rather than just trusting it.
# covariance_eigh is the right solver for the shape we have. 20000 rows but only
# 5000 columns, so the covariance matrix is 5000 by 5000 which is small and fast
# to work with. it is also exact, and I need it exact for the slicing above to
# hold. the randomized solver is quicker but only approximate.

pca = PCA(n_components=2000, svd_solver="covariance_eigh", random_state=42)
pca.fit(X)                          # fit on the train set only, never on X_test

# fitting on train and test together would be data leakage. the sklearn docs are
# blunt about it, never call fit on test data, only transform. fit learns the
# components, transform just applies them, so we learn from train and apply the
# same thing to both.
Z_train = pca.transform(X)          # (20000, 2000)
Z_test  = pca.transform(X_test)     # (6999, 2000)

cum = np.cumsum(pca.explained_variance_ratio_)
for k in [100, 500, 1000, 2000]:
    print("k=%5d  cumulative explained variance %.4f" % (k, cum[k-1]))

# From the results, it would seem that if the best 100 of the new features were kept, around 15% of the information is kept
# if best 500 are kept, around 38% of the information is kept
# if best 1000 features of kept, around 56% of the information is kept
# Even if 2000 features are kept, only 78% of the information is kept
# that is a lot lower than I expected. if the 5000 columns were all repeating the
# same information then a handful of components would cover nearly all of it. we
# need 2000 just to reach 78%, so the columns are mostly carrying their own
# separate bits of information and the data does not compress well. that fits
# with the sparsity, almost 99% of the matrix is zeros so two documents hardly
# ever share the same words.


k=  100  cumulative explained variance 0.1582
k=  500  cumulative explained variance 0.3895
k= 1000  cumulative explained variance 0.5644
k= 2000  cumulative explained variance 0.7750


In [17]:
# Verification before going any further
assert Z_train.shape == (20000, 2000)
assert Z_test.shape  == (6999, 2000)

# Checking the nesting claim on our own data rather than taking it on trust.
# if this prints true then slicing the first 100 columns really is the same as
# refitting with 100 components, and I have saved myself three extra pca fits.
# np.abs is needed because an eigenvector can come out with its sign flipped. a
# flipped component points along the same line and does not change any of the
# distances, so knn does not care, but a straight comparison would fail on it.
small = PCA(n_components=100, svd_solver="covariance_eigh", random_state=42).fit(X)
print("nested:", np.allclose(np.abs(small.transform(X)), np.abs(Z_train[:, :100])))


nested: True


In [18]:
# KNN on the PCA reduced features
# this is the actual experiment. the model is the same every single time, the
# only thing that changes is how many components we feed it, so anything we see
# in the results is down to the dimension and nothing else.
import time

for k in [2000, 1000, 500, 100]:
    t = time.time()
    knn = KNeighborsClassifier(n_neighbors=2, n_jobs=-1)
    knn.fit(Z_train[:, :k], y)          # knn does not really train, it just stores the data
    pred = knn.predict(Z_test[:, :k])   # all the work happens here, working out the distances

    pd.DataFrame({"id": test_ids, "label": pred}).to_csv(f"pca_{k}_knn.csv", index=False)
    print("k=%5d  %.0fs  predicted class-1 fraction %.4f" % (k, time.time()-t, pred.mean()))

# n_jobs = -1 uses all your cores
# the class 1 fraction is the thing to watch here. the training set is 62.5%
# class 1, so anything far below that means we are barely calling anything
# machine generated at all. with n_neighbors=2 a split vote goes to class 0, so
# we only say class 1 when both neighbours are class 1, and that drags it down.
# it drops a lot further at high k than the tie breaking on its own explains,
# which is the interesting bit and worth writing up in the report.


k= 2000  1s  predicted class-1 fraction 0.0582
k= 1000  1s  predicted class-1 fraction 0.1760
k=  500  0s  predicted class-1 fraction 0.3776
k=  100  0s  predicted class-1 fraction 0.5444


In [19]:
# Last check before uploading anything to kaggle.
# these four things are what would get a submission rejected, or worse scored
# wrongly without telling us. we only get a limited number of submissions a day
# and we need four of them for this task, so it is worth a few seconds here.
check = pd.read_csv("pca_100_knn.csv")
print(check.shape)                      # should be (6999, 2)
print(check.columns.tolist())           # should be ['id', 'label']
print(check["label"].unique())          # should only ever be 0 and 1
print(set(check["id"]) == set(pd.read_csv(DATA + "sample_submission.csv")["id"]))   # our ids have to be the ones kaggle is expecting


(6999, 2)
['id', 'label']
[1 0]
True
